In [ ]:
%pip install -q --no-deps --upgrade bitsandbytes==0.50.2
%pip install -q \
    "transformers==4.49.0" \
    "tokenizers==0.21.0" \
    "accelerate==1.4.0" \
    "sentence-transformers==3.4.1" \
    "chromadb==0.6.3"

print("Install complete. Restart the Kaggle session now, then run the notebook from the top.")

In [ ]:
import sys
import subprocess
import torch
import transformers
import tokenizers
import accelerate
import bitsandbytes as bnb
import sentence_transformers
import chromadb

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before continuing.")

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU:", gpu_name)
print("Compute capability:", capability)

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bnb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("ChromaDB:", chromadb.__version__)

# Informational only: do not fail due to unrelated Kaggle image conflicts.
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)
print("\npip check exit code:", pip_check.returncode)
print(pip_check.stdout[:12000] if pip_check.stdout else "No conflicts reported.")

In [ ]:
# Verify BitsAndBytes can execute an actual 4-bit CUDA operation.
import torch
import bitsandbytes as bnb

assert torch.cuda.is_available()
assert torch.cuda.get_device_capability(0) >= (6, 0), (
    "NF4 requires compute capability 6.0 or newer."
)

try:
    test_layer = bnb.nn.Linear4bit(
        64,
        64,
        bias=False,
        compute_dtype=torch.float16,
        quant_type="nf4"
    ).to("cuda")

    test_input = torch.randn(2, 64, device="cuda", dtype=torch.float16)
    with torch.inference_mode():
        test_output = test_layer(test_input)

    torch.cuda.synchronize()
    print("BitsAndBytes 4-bit CUDA test: SUCCESS")
    print("Test output shape:", tuple(test_output.shape))

    del test_layer, test_input, test_output
    torch.cuda.empty_cache()

except Exception as exc:
    raise RuntimeError(
        "BitsAndBytes 4-bit CUDA test failed. Stop here; do not load Mistral. "
        "Check the installed BitsAndBytes version and CUDA runtime."
    ) from exc

In [ ]:
import json
from pathlib import Path

# If auto-detection chooses the wrong file, replace this with the exact path.
DATA = None  # Example: Path("/kaggle/input/my-dataset/Bhagwad_Gita.json")

if DATA is None:
    candidates = []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates = sorted(
            p for p in input_root.rglob("*.json")
            if any(term in p.name.lower() for term in ("gita", "bhagwad", "bhagavad"))
        )
    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect the Gita JSON. Set DATA to its exact /kaggle/input/... path."
        )
    print("JSON candidates:")
    for candidate in candidates:
        print(" -", candidate)
    DATA = candidates[0]
    print("\nSelected:", DATA)

DATA = Path(DATA)
DB_DIR = Path("/kaggle/working/gita_db")
COLLECTION_NAME = "gita"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
EMBED_BATCH_SIZE = 64
CHROMA_BATCH_SIZE = 128

def load_verses(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, list):
        verses = raw
    elif isinstance(raw, dict):
        verses = next(
            (raw[key] for key in ("verses", "data", "items", "records")
             if isinstance(raw.get(key), list)),
            None,
        )
        if verses is None:
            raise ValueError(
                "JSON object must contain a list under verses/data/items/records."
            )
    else:
        raise TypeError("Unsupported JSON root. Expected a list or dictionary.")

    if not verses or not all(isinstance(row, dict) for row in verses):
        raise ValueError("Corpus must be a non-empty list of verse objects.")
    return verses

verses = load_verses(DATA)
print("Loaded verse records:", len(verses))
print("First record keys:", list(verses[0].keys()))

In [ ]:
from collections import Counter
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

def as_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()

def split_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    if not text:
        return []
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Require size > overlap >= 0.")

    if len(text) <= size:
        return [text.strip()]

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))

        if end < len(text):
            boundary = max(
                text.rfind("\n", start, end),
                text.rfind("।", start, end),
                text.rfind(". ", start, end),
                text.rfind(" ", start, end),
            )
            if boundary > start + size // 2:
                end = boundary + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break
        start = max(end - overlap, start + 1)

    return chunks

def verse_to_document(row: dict, row_index: int):
    chapter = as_text(row.get("chapter"))
    verse = as_text(row.get("verse"))
    verse_id = as_text(row.get("id")) or f"BG-{chapter}-{verse}-{row_index}"

    parts = [
        f"Chapter {chapter}, Verse {verse}",
        f"Sanskrit: {as_text(row.get('sanskrit'))}",
        f"Translation: {as_text(row.get('translation'))}",
        f"Summary: {as_text(row.get('summary'))}",
        f"Therapy context: {as_text(row.get('therapy_context'))}",
        f"Keywords: {as_text(row.get('keywords'))}",
    ]
    document = "\n".join(part for part in parts if part.strip())

    metadata = {
        "id": verse_id,
        "chapter": chapter,
        "verse": verse,
    }
    return verse_id, metadata, document

# Load the embedding model on CPU to leave GPU memory for Mistral.
embedder = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
chroma_client = chromadb.PersistentClient(
    path=str(DB_DIR),
    settings=Settings(anonymized_telemetry=False),
)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

all_ids, all_documents, all_metadatas = [], [], []

# Report repeated source verse IDs. Chunk storage IDs below remain unique
# even when the source JSON contains repeated verse IDs.
source_verse_ids = [
    as_text(row.get("id")) or f"BG-{as_text(row.get('chapter'))}-{as_text(row.get('verse'))}-{i}"
    for i, row in enumerate(verses)
]
source_id_counts = Counter(source_verse_ids)
duplicate_source_ids = {
    verse_id: count for verse_id, count in source_id_counts.items() if count > 1
}
print("Unique source verse IDs:", len(source_id_counts))
print("Repeated source verse IDs:", len(duplicate_source_ids))
if duplicate_source_ids:
    print("Examples:", list(duplicate_source_ids.items())[:15])

for row_index, row in enumerate(verses):
    verse_id, metadata, document = verse_to_document(row, row_index)
    for chunk_index, chunk in enumerate(split_text(document)):
        # Include row_index so duplicate source verse IDs cannot collide in Chroma.
        all_ids.append(f"{verse_id}::row-{row_index}::chunk-{chunk_index}")
        all_documents.append(chunk)
        all_metadatas.append(metadata.copy())

if not all_documents:
    raise ValueError("No non-empty text chunks were produced from the JSON corpus.")

for start in range(0, len(all_documents), CHROMA_BATCH_SIZE):
    end = start + CHROMA_BATCH_SIZE
    batch_docs = all_documents[start:end]
    vectors = embedder.encode(
        batch_docs,
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    collection.add(
        ids=all_ids[start:end],
        documents=batch_docs,
        metadatas=all_metadatas[start:end],
        embeddings=vectors.tolist(),
    )
    print(f"Indexed {min(end, len(all_documents))}/{len(all_documents)} chunks")

print("\nVerses loaded:", len(verses))
print("Unique source verse IDs:", len(source_id_counts))
print("Chunks indexed:", collection.count())
print("Chroma database:", DB_DIR)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

def load_mistral(model_name: str = MODEL_NAME):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for 4-bit Mistral loading.")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tokenizer, model

tokenizer, model = load_mistral()

print("Loaded model:", MODEL_NAME)
print("Embedding device:", model.get_input_embeddings().weight.device)
print("GPU allocated (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

In [ ]:
import numpy as np
import torch

SYSTEM_PROMPT = """You are a careful Bhagavad Gita philosophy assistant.
Answer compassionately and clearly using ONLY the retrieved passages.
If the passages do not support an answer, say that the available verses are insufficient.
Do not invent Sanskrit, translations, chapter numbers, verse numbers, or teachings.
Cite textual claims using the exact ID and chapter/verse metadata shown in the context.
Keep answers concise and distinguish direct textual meaning from your interpretation.
Do not present philosophical guidance as a substitute for professional medical or mental-health care."""

def retrieve_mmr(
    question: str,
    k: int = 8,
    fetch_k: int = 48,
    lambda_mult: float = 0.5,
):
    if not isinstance(question, str) or not question.strip():
        return []
    if collection.count() == 0:
        return []

    query_vector = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0].astype(np.float32)

    candidates = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=min(fetch_k, collection.count()),
        include=["documents", "metadatas", "embeddings"],
    )

    docs = candidates["documents"][0]
    metas = candidates["metadatas"][0]
    vectors = np.asarray(candidates["embeddings"][0], dtype=np.float32)

    if not docs:
        return []

    relevance = vectors @ query_vector
    selected_indices = []
    remaining = set(range(len(docs)))
    selected_verse_ids = set()

    while remaining and len(selected_indices) < k:
        best_idx = None
        best_score = -float("inf")

        for idx in remaining:
            if selected_indices:
                redundancy = max(
                    float(vectors[idx] @ vectors[j])
                    for j in selected_indices
                )
            else:
                redundancy = 0.0

            score = (
                lambda_mult * float(relevance[idx])
                - (1.0 - lambda_mult) * redundancy
            )
            if score > best_score:
                best_idx, best_score = idx, score

        remaining.remove(best_idx)
        verse_id = str(metas[best_idx].get("id", ""))

        if verse_id in selected_verse_ids:
            continue

        selected_verse_ids.add(verse_id)
        selected_indices.append(best_idx)

    return [
        {
            "id": str(metas[i].get("id", "")),
            "chapter": str(metas[i].get("chapter", "")),
            "verse": str(metas[i].get("verse", "")),
            "text": docs[i],
        }
        for i in selected_indices
    ]

def format_context(passages: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[ID: {p['id']} | Chapter: {p['chapter']} | Verse: {p['verse']}]\n{p['text']}"
        for p in passages
    )

def generate_answer(
    question: str,
    passages: list[dict],
    chat_history: list[dict] | None = None,
    max_new_tokens: int = 300,
) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for turn in (chat_history or [])[-6:]:
        role = turn.get("role")
        content = turn.get("content", "")
        if role in {"user", "assistant"} and isinstance(content, str) and content.strip():
            messages.append({"role": role, "content": content})

    messages.append({
        "role": "user",
        "content": (
            f"Retrieved passages:\n{format_context(passages)}\n\n"
            f"Question: {question}\n\n"
            "Answer using the passages and cite the exact verse IDs."
        ),
    })

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=5000,
    )

    input_device = model.get_input_embeddings().weight.device
    encoded = {key: value.to(input_device) for key, value in encoded.items()}

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0, encoded["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

def ask_gita(
    question: str,
    chat_history: list[dict] | None = None,
    show_sources: bool = True,
) -> str:
    passages = retrieve_mmr(question)

    if not passages:
        return "I could not retrieve relevant verses from the indexed corpus."

    answer = generate_answer(question, passages, chat_history)

    if show_sources:
        source_lines = [
            f"- Chapter {p['chapter']}, Verse {p['verse']} (ID: {p['id']})"
            for p in passages
        ]
        answer += "\n\nRetrieved sources:\n" + "\n".join(source_lines)

    return answer

print("RAG functions are ready.")

In [ ]:
test_question = "What is the meaning of life"

test_passages = retrieve_mmr(test_question, k=8, fetch_k=48)
print("Retrieved distinct verses:", len(test_passages))

for index, passage in enumerate(test_passages, start=1):
    print(f"\n[{index}] Chapter {passage['chapter']}, Verse {passage['verse']} | ID: {passage['id']}")
    print(passage["text"][:1000])

In [ ]:
print(ask_gita(test_question))

In [ ]:
chat_history = []

print("Bhagavad Gita RAG assistant. Type 'exit' to stop.")

while True:
    try:
        question = input("\nYou: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if question.lower() in {"exit", "quit"}:
        print("Goodbye.")
        break

    if not question:
        continue

    try:
        response = ask_gita(question, chat_history)
        print("\nAssistant:", response)

        chat_history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": response},
        ])
        chat_history = chat_history[-8:]

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("CUDA ran out of memory. Try reducing max_new_tokens or the prompt length.")
    except Exception as exc:
        print(f"Request failed: {type(exc).__name__}: {exc}")